# 97 - Analyze SmolVLA stock/refinement consensus projection

Compact, zero-GPU analysis of notebook 96. It supports a partially completed run, always uses exact identity matches, and reports overall/per-suite SR changes, outcome flips, and failure/flip AUCs for parent and projection uncertainty. Projection uncertainty is available before executing the projected chunk, but whole-episode averages remain retrospective.


In [ ]:
EXTRAS = 'analysis'
SETUP_ENV = False
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

In [ ]:
from pathlib import Path
import hashlib, io, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

from analysis.horizon_diagnostics import failure_auc_table, _download_with_retry
from analysis.smolvla_libero import (
    TRANSITION_ORDER, flip_auc_table, pair_stock_refinement,
    uncertainty_quantile_flip_table)
from analysis.suffix_sensitivity import summarize_pair
from pnp.diversity import DIVERSITY_PAIR_KEYS
from pnp.experiments import (
    SMOLVLA_LIBERO_EXPERIMENT, build_smolvla_libero_methods)
from pnp.smolvla_followup_experiments import (
    SMOLVLA_SCHEDULE_EXPERIMENT, build_smolvla_schedule_method)
from pnp.smolvla_consensus_projection_experiment import (
    SMOLVLA_CONSENSUS_PROJECTION_EXPERIMENT,
    build_smolvla_consensus_projection_methods)
from pnp.store import SupabaseStore

HORIZONS = (10, 20, 50)
N_BOOT = 2000
OUTPUT = Path('smolvla_consensus_projection_analysis')
CACHE = OUTPUT / 'cache'
OUTPUT.mkdir(exist_ok=True); CACHE.mkdir(exist_ok=True)
store = SupabaseStore()

def completed(experiment, method, config):
    config_hash = store.config_hash(store._logical_key(method, config))
    frame = pd.DataFrame(store.fetch_all(
        'rollouts', '*', configure=lambda query: query.eq('experiment', experiment)
        .eq('method', method).eq('config_hash', config_hash).eq('status', 'completed'),
        order_by=('rollout_id',)))
    if len(frame) and frame.duplicated(DIVERSITY_PAIR_KEYS).any():
        raise ValueError(f'duplicate identities for {experiment}/{method}')
    return frame, config_hash

stock_method, stock_config = build_smolvla_libero_methods()[0]
stock, stock_hash = completed(SMOLVLA_LIBERO_EXPERIMENT, stock_method, stock_config)
refine_method, refine_config = build_smolvla_schedule_method()
historic_refine, refine_hash = completed(
    SMOLVLA_SCHEDULE_EXPERIMENT, refine_method, refine_config)
projection_configs = dict(build_smolvla_consensus_projection_methods())
projection_rows, projection_hashes = {}, {}
for method, config in projection_configs.items():
    projection_rows[method], projection_hashes[method] = completed(
        SMOLVLA_CONSENSUS_PROJECTION_EXPERIMENT, method, config)
    if not len(projection_rows[method]):
        raise ValueError(f'no completed rows found for {method}')

def exact_pair(baseline, condition):
    keys = condition[DIVERSITY_PAIR_KEYS].drop_duplicates()
    matched = baseline.merge(keys, on=DIVERSITY_PAIR_KEYS, validate='one_to_one')
    return pair_stock_refinement(matched, condition)

pairs = {method: exact_pair(stock, rows) for method, rows in projection_rows.items()}
pairs['historic_refine'] = exact_pair(stock, historic_refine)
coverage = pd.DataFrame([{
    'arm': method, 'completed': len(rows), 'successes': int(rows.success.sum()),
    'sr_pct': 100 * rows.success.mean(), 'config_hash': projection_hashes[method],
    'uncertainty_artifacts': int(rows.ahats_path.notna().sum())}
    for method, rows in projection_rows.items()])
print({'experiment': SMOLVLA_CONSENSUS_PROJECTION_EXPERIMENT,
       'expected_full_identities_per_arm': 400, 'historical_stock_rows': len(stock)})
display(coverage)


## Exact-matched SR and outcome transitions


In [ ]:
overall_tables, suite_tables = [], []
for arm, paired in pairs.items():
    overall, by_suite = summarize_pair(paired)
    overall.insert(0, 'arm', arm); by_suite.insert(0, 'arm', arm)
    overall_tables.append(overall); suite_tables.append(by_suite)
overall = pd.concat(overall_tables, ignore_index=True)
by_suite = pd.concat(suite_tables, ignore_index=True)
display(overall)
display(by_suite)
overall.to_csv(OUTPUT / 'overall_paired_sr.csv', index=False)
by_suite.to_csv(OUTPUT / 'suite_paired_sr.csv', index=False)

transition_rows = []
for arm, paired in pairs.items():
    counts = paired.transition.value_counts().reindex(TRANSITION_ORDER, fill_value=0)
    transition_rows.append({'arm': arm, 'episodes': len(paired),
        **{name: int(counts[name]) for name in TRANSITION_ORDER},
        'net_gain_pp': 100*(counts['F->S']-counts['S->F'])/len(paired)})
transitions = pd.DataFrame(transition_rows)
display(transitions)
transitions.to_csv(OUTPUT / 'overall_transitions.csv', index=False)

plot = by_suite[by_suite.arm.ne('historic_refine')].copy()
suites = sorted(plot.suite.unique()); arms = list(projection_rows)
x = np.arange(len(suites)); width = .36
fig, ax = plt.subplots(figsize=(13, 5))
for offset, arm, color in zip((-.18, .18), arms, ('#4C78A8', '#F58518')):
    group = plot[plot.arm.eq(arm)].set_index('suite').reindex(suites)
    ax.bar(x+offset, group.condition_minus_baseline_pp, width, label=arm, color=color)
ax.axhline(0, color='black'); ax.set_xticks(
    x, [s.removeprefix('libero_') for s in suites], rotation=25, ha='right')
ax.set(ylabel='Projection minus stock SR (percentage points)',
       title='Exact-matched SR change by suite')
ax.legend(); ax.grid(axis='y', alpha=.2)
fig.tight_layout(); fig.savefig(OUTPUT / 'sr_delta_by_suite.png', dpi=180); plt.show()


## Parent versus projection uncertainty

`parent_weighted` combines the parent refinement's steps 1/2/3 with weights 3/1/1. `projection` is the fixed-time P&P diagnostic at step 5 (`s=0.5`).


In [ ]:
KEY = re.compile(r'^c(?P<chunk>\d+)_s(?P<step>\d+)_u_time$')

def decode_arm(method, rows):
    signature = hashlib.sha1('\n'.join(sorted(rows.rollout_id)).encode()).hexdigest()[:12]
    path = CACHE / f'{method}_{signature}.pkl'
    if path.exists():
        return pd.read_pickle(path)
    decoded = []
    for row in tqdm(rows.to_dict('records'), desc=f'{method} artifacts'):
        if not row.get('ahats_path'):
            raise ValueError(f'{row["rollout_id"]} has no uncertainty artifact')
        payload = _download_with_retry(store, str(row['ahats_path']))
        with np.load(io.BytesIO(payload)) as archive:
            for key in archive.files:
                match = KEY.match(key)
                if not match:
                    continue
                step = int(match.group('step'))
                if step not in (1, 2, 3, 5):
                    continue
                profile = np.asarray(archive[key], dtype=float).reshape(-1)
                item = {'rollout_id': row['rollout_id'], 'chunk_idx': int(match.group('chunk')),
                        'euler_step': step}
                for horizon in HORIZONS:
                    item[f'u{horizon}'] = float(profile[:horizon].mean())
                decoded.append(item)
    result = pd.DataFrame(decoded)
    result.to_pickle(path)
    return result

def make_features(rows, records):
    metrics = [f'u{h}' for h in HORIZONS]
    wide = records.set_index(['rollout_id', 'chunk_idx', 'euler_step'])[metrics].unstack('euler_step')
    if not {1,2,3,5}.issubset(set(wide.columns.get_level_values(1))):
        raise ValueError('at least one projection rollout lacks steps 1,2,3,5')
    chunks = pd.DataFrame(index=wide.index)
    for horizon in HORIZONS:
        chunks[f'u{horizon}_parent_weighted'] = (
            3*wide[(f'u{horizon}',1)] + wide[(f'u{horizon}',2)] + wide[(f'u{horizon}',3)]) / 5
        chunks[f'u{horizon}_projection'] = wide[(f'u{horizon}',5)]
    chunks = chunks.reset_index()
    values = [c for c in chunks if c.startswith('u')]
    episode = chunks.groupby('rollout_id')[values].mean()
    episode.columns = [f'{c}_episode' for c in episode]
    first = chunks[chunks.chunk_idx.eq(0)].set_index('rollout_id')[values]
    first.columns = [f'{c}_first_chunk' for c in first]
    metadata = rows[DIVERSITY_PAIR_KEYS + ['rollout_id','success']].set_index('rollout_id')
    return metadata.join(episode).join(first).reset_index(), chunks

features, chunk_tables = {}, {}
for method, rows in projection_rows.items():
    records = decode_arm(method, rows)
    features[method], chunk_tables[method] = make_features(rows, records)
    print(method, {'episodes': len(features[method]), 'chunks': len(chunk_tables[method])})


In [ ]:
auc_tables, flip_tables = [], []
for method in projection_rows:
    scores = [f'u{h}_{stage}_{timing}' for h in HORIZONS
              for stage in ('parent_weighted','projection')
              for timing in ('episode','first_chunk')]
    auc = failure_auc_table(features[method], scores, n_boot=N_BOOT)
    auc.insert(0, 'arm', method); auc_tables.append(auc)
    scored = pairs[method].merge(
        features[method][DIVERSITY_PAIR_KEYS + scores],
        on=DIVERSITY_PAIR_KEYS, validate='one_to_one')
    flips = flip_auc_table(scored, scores, n_boot=N_BOOT)
    flips.insert(0, 'arm', method); flip_tables.append(flips)
aucs = pd.concat(auc_tables, ignore_index=True)
flips = pd.concat(flip_tables, ignore_index=True)
print('Pooled failure AUC')
display(aucs[aucs.suite.eq('pooled')])
print('Per-suite failure AUC')
display(aucs[~aucs.suite.eq('pooled')])
print('Does uncertainty predict any flip, rescue, or harm?')
display(flips)
aucs.to_csv(OUTPUT / 'uncertainty_failure_auc.csv', index=False)
flips.to_csv(OUTPUT / 'uncertainty_flip_auc.csv', index=False)

k3 = list(projection_rows)[0]
k3_scores = ['u10_parent_weighted_first_chunk','u10_projection_first_chunk',
             'u10_parent_weighted_episode','u10_projection_episode']
k3_scored = pairs[k3].merge(
    features[k3][DIVERSITY_PAIR_KEYS + k3_scores],
    on=DIVERSITY_PAIR_KEYS, validate='one_to_one')
quantiles = pd.concat([
    uncertainty_quantile_flip_table(k3_scored, score_column=score, bins=4)
    for score in k3_scores], ignore_index=True)
display(quantiles)
quantiles.to_csv(OUTPUT / 'k3_uncertainty_quantile_flips.csv', index=False)

pooled = aucs[aucs.suite.eq('pooled')].copy()
plot_scores = [f'u{h}_projection_episode' for h in HORIZONS]
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
for arm, color in zip(projection_rows, ('#4C78A8','#F58518')):
    group = pooled[pooled.arm.eq(arm)].set_index('score_name').reindex(plot_scores)
    axes[0].plot(HORIZONS, group.failure_auc, marker='o', label=arm, color=color)
axes[0].axhline(.5, color='black', linestyle='--')
axes[0].set(xlabel='Action horizon', ylabel='Failure ROC-AUC',
            title='Projection uncertainty predicts arm failure')
axes[0].legend(); axes[0].grid(alpha=.2)
k3_suite = aucs[(aucs.arm.eq(k3)) & ~aucs.suite.eq('pooled')
                 & aucs.score_name.eq('u10_projection_episode')].sort_values('suite')
axes[1].errorbar(k3_suite.failure_auc, np.arange(len(k3_suite)),
    xerr=np.vstack((k3_suite.failure_auc-k3_suite.auc_ci_low,
                    k3_suite.auc_ci_high-k3_suite.failure_auc)), fmt='o', capsize=3)
axes[1].set_yticks(np.arange(len(k3_suite)),
    k3_suite.suite.str.removeprefix('libero_'))
axes[1].axvline(.5, color='black', linestyle='--')
axes[1].set(xlim=(0,1), xlabel='Failure ROC-AUC', title='K=3 projection U10 by suite')
axes[1].grid(axis='x', alpha=.2)
fig.savefig(OUTPUT / 'projection_uncertainty_auc.png', dpi=180); plt.show()
print('Outputs:', OUTPUT.resolve())
